# NeuroScan AI: Preprocesamiento y Pipeline tf.data
## Módulo 3: Databricks e IA Aplicada - Proyecto Final

Este notebook implementa la **Etapa 2**:
1. Construcción de canales estandarizados `(224, 224, 3)`.
2. Normalización de pesos ImageNet (BGR mean centering).
3. Data Augmentation clínicamente seguro para neuroimagen.
4. Optimización de pipeline con `prefetch(AUTOTUNE)` y verificación de lotes.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import BATCH_SIZE, CLASSES, IMG_CHANNELS, IMG_SIZE, NUM_CLASSES, SPLIT_MANIFEST_PATH
from src.preprocessing import create_tf_dataset

print(f"TensorFlow Version: {tf.__version__}")
print(f"Dispositivos disponibles: {tf.config.list_physical_devices()}")


### 1. Cargar el Split de Datos


In [ ]:
df_split = pd.read_csv(SPLIT_MANIFEST_PATH)
print("Resumen de splits:")
print(df_split.groupby(["split", "label_name"]).size().unstack())


### 2. Construir Pipelines tf.data (Train, Val, Test)


In [ ]:
train_ds = create_tf_dataset(df_split, split_name="train", batch_size=BATCH_SIZE, model_type="resnet50")
val_ds = create_tf_dataset(df_split, split_name="val", batch_size=BATCH_SIZE, model_type="resnet50")
test_ds = create_tf_dataset(df_split, split_name="test", batch_size=BATCH_SIZE, model_type="resnet50")

print(f"Pipeline construido: Lotes de tamaño {BATCH_SIZE} listos para GPU/CPU.")


### 3. Inspección Visual de un Lote


In [ ]:
for images, labels in train_ds.take(1):
    print(f"Tensor de imágenes: {images.shape}, tipo: {images.dtype}")
    print(f"Tensor de etiquetas: {labels.shape}, tipo: {labels.dtype}")
    
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for i, ax in enumerate(axes.flat):
        img_np = images[i].numpy()
        # Deshacer centrado de media para visualización legible
        img_disp = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-6)
        label_idx = np.argmax(labels[i].numpy())
        ax.imshow(img_disp)
        ax.set_title(f"Clase: {CLASSES[label_idx]}")
        ax.axis("off")
    plt.suptitle("Muestras de Resonancias Preprocesadas con Data Augmentation")
    plt.tight_layout()
    plt.show()
